# Análisis de matrices de atención con BERT multilingüe


[Link Repositorio](https://github.com/donmatthiuz/NLP/tree/lab3)


## Instalación e importaciones

In [1]:
!pip install -q transformers torch pandas

error: externally-managed-environment

× This environment is externally managed
╰─> This Python installation is managed by uv and should not be modified.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [2]:
import torch
import pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModel

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
torch.set_grad_enabled(False)   # solo inferencia: no necesitamos gradientes

NOMBRE_MODELO = "bert-base-multilingual-cased"

print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
print("pandas      :", pd.__version__)

/home/donmathiuz/.cache/uv/archive-v0/vTMscW0S9HaOEqJl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.16.1
torch       : 2.14.0+cu130
pandas      : 3.0.5


---
## 1. Corpus

### 1.1 Selección de las páginas

 Se eligió el inicio del par consecutivo al azar dentro del cuerpo del texto (páginas 3–42, para no caer en portada ni contraportada):

```python
import random
random.seed(20260831)
inicio = random.randint(3, 41)   
paginas = (inicio, inicio + 1)   
```

El texto se extrajo con `pdftotext -f 15 -l 16 -layout` (usando en local con nix-shell -p) y se reunieron las líneas en párrafos y se guardo en un directorio `corpus/metamorfosis_p15-16.txt` para luego copiarlo al notebook directamente.

In [3]:
CORPUS = """de su excitación, no se le ocurrió abrir la otra hoja para dejar espacio suficiente. Estaba obsesionado con la idea de que Gregorio había de meterse cuanto antes en su habitación. Tampoco hubiera permitido los lentos preparativos que Gregorio necesitaba para incorporarse y, de este modo, pasar por la puerta. Como si no hubiese problema alguno azuzaba a Gregorio con furia creciente. Gregorio oía tras de sí una voz que parecía imposible que fuese la de un padre. Se incrustó en el marco de la puerta. Se irguió de medio lado y quedó atravesado en el umbral, lacerándose el costado. En la puerta aparecieron unas manchas repulsivas. Gregorio quedó allí atascado, sin posibilidad de hacer el menor movimiento.

Las patitas de uno de los lados colgaban en el aire, mientras que las del otro quedaban dolorosamente oprimidas contra el suelo... En esto, el padre le dio por detrás un empujón enérgico y salvador, que lo lanzó dentro del cuarto, sangrando copiosamente. Luego, cerró la puerta con el bastón, y por fin volvió a la calma.

Hasta la noche no despertó Gregorio de un pesado sueño, semejante a un desmayo. No habría tardado mucho en despabilarse por sí solo, pues ya había descansado bastante, pero le pareció que le despertaban unos pasos furtivos y el ruido de la puerta del recibidor, que alguien cerraba suavemente. El reflejo del tranvía proyectaba franjas de luz en el techo de la habitación y la parte superior de los muebles; pero de abajo, donde estaba Gregorio, reinaba la oscuridad. Lenta y todavía torpemente, tanteando con sus antenas, que en ese momento le mostraron su utilidad, se deslizó hacia la puerta para ver lo que había ocurrido. En su costado izquierdo había una larga y repugnante llaga. Renqueaba alternativamente sobre cada una de sus dos hileras de patas, una de las cuales herida en el accidente de la mañana — sorprendentemente, las demás habían quedado ilesas—, se arrastraba sin vida.

Al llegar a la puerta, comprendió que lo que le había atraído era el olor de algo comestible. Encontró una cazoleta llena de leche con azúcar, en la que flotaban trocitos de pan. Estuvo a punto de reír de gozo, pues tenía aún más hambre que por la mañana. Hundió la cabeza en la leche casi hasta los ojos; pero enseguida la retiró contrariado, pues no sólo la herida de su costado izquierdo le hacía dificultosa la operación (para comer tenía que mover todo el cuerpo), sino que, además, la leche, que hasta entonces había sido su bebida predilecta —por eso, sin duda, la había puesto allí su hermana—, no le gustó nada. Se apartó casi con repugnancia de la cazoleta y se arrastró de nuevo hacia el centro de la habitación. Por la rendija de la puerta vio que la luz estaba encendida en el comedor. Pero, en contra de lo habitual, no se oía al padre leer en voz alta a la madre y la hermana el diario de la tarde. No se oía el menor ruido. Quizá esta costumbre, de la que siempre le hablaba la hermana en sus cartas, hubiese desaparecido. Todo estaba silencioso, pese a que, con toda seguridad, la casa no estaba vacía. «¡Qué vida tan tranquila lleva mi familia!», pensó Gregorio. Mientras su mirada se perdía en las sombras, se sintió orgulloso de haber podido proporcionar a sus padres y a su hermana tan sosegada existencia, en un hogar tan acogedor. De pronto pensó con terror que aquella tranquilidad, aquel bienestar y aquella alegría iban a terminar... Para no abandonarse en estos pensamientos, prefirió ponerse en movimiento y comenzó a arrastrarse por la habitación.

Durante la noche se entreabrió una vez una de las hojas de la puerta, y otra vez la otra: alguien quería entrar. Gregorio, en vista de ello, se colocó contra la puerta que daba al comedor, dispuesto a atraer hacia el interior al indeciso visitante, o por lo menos a averiguar quién era. Pero la puerta no volvió a abrirse, y esperó en vano. Esa mañana, cuando la puerta estaba cerrada, todos habían intentado entrar, y ahora que él había abierto una puerta y que la otra había sido también abierta, sin duda, durante el día, ya no venía nadie, y las llaves habían sido puestas en la parte exterior de las cerraduras.

Estaba muy avanzada la noche cuando se apagó la luz del comedor. Gregorio comprendió que sus padres habían permanecido en vela hasta entonces. Oyó como se alejaban de puntillas. Hasta la mañana no entraría seguramente nadie a ver a Gregorio: tenía tiempo de sobra para pensar, sin temor a ser importunado, en su futuro. Pero aquella habitación fría y de techo alto, en donde había de permanecer echado de bruces. Le dio miedo; no entendía por qué, pues era la suya, la habitación en que vivía desde hacía cinco años... Bruscamente, y no sin algo de vergüenza, se metió debajo del sofá, en donde, a pesar de sentirse algo estrujado, por no poder levantar la cabeza, se encontró en seguida muy bien, lamentando únicamente no poder introducirse allí por completo a causa de su excesiva corpulencia.

Así permaneció toda la noche, sumido en un duermevela del que le despertaba con sobresalto el hambre, y sacudido por preocupaciones y esperanzas no muy concretas, pero cuya conclusión era siempre la necesidad de tener calma y paciencia y de hacer lo posible para que su familia se hiciese cargo de la situación y no sufriera más de lo necesario.

Muy temprano, cuando apenas empezaba a clarear, Gregorio tuvo ocasión de poner en práctica sus resoluciones. Su hermana, ya casi arreglada, abrió la puerta que daba al recibidor y le buscó ansiosamente con la mirada. Al principio no le vio; pero al descubrirle debajo del sofá —¡en algún sitio había de estar! ¡No iba a haber volado!— se asustó tanto que, compulsivamente, volvió a cerrar la puerta."""

parrafos = [p for p in CORPUS.split("\n\n") if p.strip()]
print(f"Párrafos : {len(parrafos)}")
print(f"Palabras : {len(CORPUS.split())}")
print(f"Caracteres: {len(CORPUS)}")
print()
print("Primer párrafo:")
print(parrafos[0][:300], "...")

Párrafos : 8
Palabras : 981
Caracteres: 5665

Primer párrafo:
de su excitación, no se le ocurrió abrir la otra hoja para dejar espacio suficiente. Estaba obsesionado con la idea de que Gregorio había de meterse cuanto antes en su habitación. Tampoco hubiera permitido los lentos preparativos que Gregorio necesitaba para incorporarse y, de este modo, pasar por l ...


### 1.2 Oraciones seleccionadas

Se eligen seis oraciones del corpus con propósitos distintos, más una oración **control** externa para el contraste de polisemia:

| id | Por qué se eligió |
|---|---|
| `S1_simple` | Oración simple, un solo verbo, sin subordinación. Línea base. |
| `S2_coord` | Coordinación + gerundio adjunto (`lacerándose`), palabra que el tokenizador partirá. |
| `S3_sub` | Doble subordinación encadenada (`una voz que parecía imposible que fuese...`): dependencia de larga distancia. |
| `S4_larga` | Oración larga con inciso y relativo: caso de máxima complejidad sintáctica. |
| `S5_hoja_puerta` | `hoja` = hoja de puerta (sentido arquitectónico). |
| `S6_hojas_puerta` | `hojas` de nuevo como parte de la puerta, en otro contexto del mismo corpus. |
| `S7_hoja_control` | **Fuera del corpus.** `hoja` = hoja vegetal. Sirve de contraste léxico al estilo del ejemplo *banco/banco* del enunciado. |

Los **tokens objetivo** (`objetivos`) son las palabras desde las que se leerán las filas de la matriz de atención en la Parte C: sustantivos y verbos con carga sintáctica, no artículos ni preposiciones.

In [4]:
ORACIONES = {
    "S1_simple": {
        "texto": "Se incrustó en el marco de la puerta.",
        "objetivos": ["incrustó", "puerta"],
        "nota": "Simple: un verbo, sin subordinación.",
    },
    "S2_coord": {
        "texto": "Se irguió de medio lado y quedó atravesado en el umbral, lacerándose el costado.",
        "objetivos": ["quedó", "umbral", "lacerándose"],
        "nota": "Coordinada + gerundio adjunto; 'lacerándose' se fragmenta en subpalabras.",
    },
    "S3_sub": {
        "texto": "Gregorio oía tras de sí una voz que parecía imposible que fuese la de un padre.",
        "objetivos": ["voz", "padre", "oía"],
        "nota": "Relativa + completiva encadenadas: dependencia de larga distancia.",
    },
    "S4_larga": {
        "texto": (
            "Renqueaba alternativamente sobre cada una de sus dos hileras de patas, "
            "una de las cuales herida en el accidente de la mañana — sorprendentemente, "
            "las demás habían quedado ilesas—, se arrastraba sin vida."
        ),
        "objetivos": ["patas", "herida", "Renqueaba"],
        "nota": "Larga, con inciso parentético y relativo partitivo: máxima distancia sujeto-verbo.",
    },
    "S5_hoja_puerta": {
        "texto": "No se le ocurrió abrir la otra hoja para dejar espacio suficiente.",
        "objetivos": ["hoja", "abrir"],
        "nota": "'hoja' = batiente de la puerta. Contexto: abrir / espacio.",
    },
    "S6_hojas_puerta": {
        "texto": "Durante la noche se entreabrió una vez una de las hojas de la puerta.",
        "objetivos": ["hojas", "puerta"],
        "nota": "'hojas' = batientes, ahora con 'puerta' explícita en la oración.",
    },
    "S7_hoja_control": {
        "texto": "La hoja del árbol cayó sobre el suelo del jardín.",
        "objetivos": ["hoja", "árbol"],
        "nota": "CONTROL (fuera del corpus): 'hoja' = hoja vegetal.",
    },
}

# Verificación: las oraciones S1-S6 deben aparecer literalmente en el corpus.
import re

def normalizar(s):
    return re.sub(r"[\s,]+", " ", s).strip().lower()

corpus_norm = normalizar(CORPUS)
filas = []
for oid, info in ORACIONES.items():
    control = oid.endswith("_control")
    en_corpus = normalizar(info["texto"]).rstrip(".") in corpus_norm
    filas.append({
        "id": oid,
        "n_palabras": len(info["texto"].split()),
        "en_corpus": "—(control)" if control else ("sí" if en_corpus else "NO"),
        "objetivos": ", ".join(info["objetivos"]),
        "texto": info["texto"],
    })

pd.DataFrame(filas).set_index("id")

,n_palabras,en_corpus,objetivos,texto
id,,,,
S1_simple,8,sí,"incrustó, puerta",Se incrustó en el marco de la puerta.
S2_coord,14,sí,"quedó, umbral, lacerándose","Se irguió de medio lado y quedó atravesado en el umbral, lacerándose el costado."
S3_sub,16,sí,"voz, padre, oía",Gregorio oía tras de sí una voz que parecía imposible que fuese la de un padre.
S4_larga,33,sí,"patas, herida, Renqueaba","Renqueaba alternativamente sobre cada una de sus dos hileras de patas, una de las cuales herida en el accidente de la mañana — sorprendentemente, las demás habían quedado ilesas—, se arrastraba sin vida."
S5_hoja_puerta,12,sí,"hoja, abrir",No se le ocurrió abrir la otra hoja para dejar espacio suficiente.
S6_hojas_puerta,14,sí,"hojas, puerta",Durante la noche se entreabrió una vez una de las hojas de la puerta.
S7_hoja_control,10,—(control),"hoja, árbol",La hoja del árbol cayó sobre el suelo del jardín.


---
# Parte A — Tokenización

Objetivo: ver qué le entra realmente al modelo. Nos interesan tres cosas:

1. Los **tokens especiales** `[CLS]` (posición 0, resumen de secuencia) y `[SEP]` (final).
2. Las **subpalabras**: WordPiece marca la continuación de una palabra con el prefijo `##`.
3. La **diferencia entre palabra lingüística y token del modelo**: no hay correspondencia 1:1.

In [5]:
tokenizador = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

print("Tipo de tokenizador:", type(tokenizador).__name__)
print("Tamaño del vocabulario:", tokenizador.vocab_size)
print("Tokens especiales:", tokenizador.all_special_tokens)
print("IDs especiales   :", tokenizador.all_special_ids)
print("Longitud máxima  :", tokenizador.model_max_length)

Tipo de tokenizador: BertTokenizer
Tamaño del vocabulario: 119547
Tokens especiales: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']
IDs especiales   : [100, 102, 0, 101, 103]
Longitud máxima  : 512


In [6]:
def tabla_tokens(texto):
    """Devuelve un DataFrame con posición, token, id y tipo de cada token."""
    cod = tokenizador(texto, return_tensors="pt")
    ids = cod["input_ids"][0].tolist()
    toks = tokenizador.convert_ids_to_tokens(ids)
    tipos = []
    for t in toks:
        if t in tokenizador.all_special_tokens:
            tipos.append("especial")
        elif t.startswith("##"):
            tipos.append("subpalabra (##)")
        else:
            tipos.append("palabra/inicio")
    return pd.DataFrame({"pos": range(len(toks)), "token": toks, "id": ids, "tipo": tipos})


for oid, info in ORACIONES.items():
    print("=" * 90)
    print(f"{oid}  —  {info['nota']}")
    print(f"  «{info['texto']}»")
    toks = tokenizador.tokenize(info["texto"])
    print(f"  palabras (split): {len(info['texto'].split()):>3}   "
          f"tokens WordPiece: {len(toks):>3}   "
          f"+ especiales: {len(toks) + 2}")
    print("  tokens:", toks)
    print()

S1_simple  —  Simple: un verbo, sin subordinación.
  «Se incrustó en el marco de la puerta.»
  palabras (split):   8   tokens WordPiece:  12   + especiales: 14
  tokens: ['Se', 'in', '##c', '##rust', '##ó', 'en', 'el', 'marco', 'de', 'la', 'puerta', '.']

S2_coord  —  Coordinada + gerundio adjunto; 'lacerándose' se fragmenta en subpalabras.
  «Se irguió de medio lado y quedó atravesado en el umbral, lacerándose el costado.»
  palabras (split):  14   tokens WordPiece:  23   + especiales: 25
  tokens: ['Se', 'ir', '##gui', '##ó', 'de', 'medio', 'lado', 'y', 'quedó', 'at', '##rave', '##sado', 'en', 'el', 'umbral', ',', 'lac', '##er', '##ándose', 'el', 'costa', '##do', '.']

S3_sub  —  Relativa + completiva encadenadas: dependencia de larga distancia.
  «Gregorio oía tras de sí una voz que parecía imposible que fuese la de un padre.»
  palabras (split):  16   tokens WordPiece:  20   + especiales: 22
  tokens: ['Gregorio', 'o', '##ía', 'tras', 'de', 'sí', 'una', 'voz', 'que', 'pare', '##cía

In [7]:
# Vista detallada de una oración: aquí se ven [CLS], [SEP] y las subpalabras ##
tabla_tokens(ORACIONES["S2_coord"]["texto"])

,pos,token,id,tipo
0,0,[CLS],101,especial
1,1,Se,11045,palabra/inicio
2,2,ir,10478,palabra/inicio
3,3,##gui,55818,subpalabra (##)
4,4,##ó,10443,subpalabra (##)
5,5,de,10104,palabra/inicio
6,6,medio,15762,palabra/inicio
7,7,lado,15776,palabra/inicio
8,8,y,193,palabra/inicio
9,9,quedó,30801,palabra/inicio


In [8]:
# A.3 — Palabra lingüística vs. token del modelo: ¿qué palabras se fragmentan?
filas = []
for oid, info in ORACIONES.items():
    for palabra in info["texto"].split():
        limpia = palabra.strip(".,;:()«»¡!¿?…")
        if not limpia:
            continue
        piezas = tokenizador.tokenize(limpia)
        if len(piezas) > 1:
            filas.append({
                "oración": oid,
                "palabra": limpia,
                "n_tokens": len(piezas),
                "subpalabras": " | ".join(piezas),
            })

df_frag = pd.DataFrame(filas).drop_duplicates(subset="palabra").sort_values(
    "n_tokens", ascending=False
).reset_index(drop=True)
print(f"Palabras fragmentadas en subpalabras: {len(df_frag)}")
df_frag

Palabras fragmentadas en subpalabras: 18


,oración,palabra,n_tokens,subpalabras
0,S1_simple,incrustó,4,in | ##c | ##rust | ##ó
1,S2_coord,irguió,3,ir | ##gui | ##ó
2,S2_coord,atravesado,3,at | ##rave | ##sado
3,S2_coord,lacerándose,3,lac | ##er | ##ándose
4,S4_larga,Renqueaba,3,Ren | ##que | ##aba
5,S4_larga,sorprendentemente,3,sor | ##prende | ##ntemente
6,S4_larga,ilesas—,3,ile | ##sas | [UNK]
7,S4_larga,arrastraba,3,arra | ##stra | ##ba
8,S6_hojas_puerta,entreabrió,3,entre | ##ab | ##rió
9,S2_coord,costado,2,costa | ##do


In [9]:
# A.4 — Resumen cuantitativo: fertilidad del tokenizador (tokens por palabra)
resumen = []
for oid, info in ORACIONES.items():
    toks = tokenizador.tokenize(info["texto"])
    n_pal = len(info["texto"].split())
    n_sub = sum(1 for t in toks if t.startswith("##"))
    resumen.append({
        "oración": oid,
        "palabras": n_pal,
        "tokens": len(toks),
        "tokens+especiales": len(toks) + 2,
        "subpalabras ##": n_sub,
        "tokens/palabra": round(len(toks) / n_pal, 2),
    })

df_resumen = pd.DataFrame(resumen).set_index("oración")
df_resumen

,palabras,tokens,tokens+especiales,subpalabras ##,tokens/palabra
oración,,,,,
S1_simple,8,12,14,3,1.50
S2_coord,14,23,25,7,1.64
S3_sub,16,20,22,3,1.25
S4_larga,33,49,51,11,1.48
S5_hoja_puerta,12,14,16,1,1.17
S6_hojas_puerta,14,17,19,2,1.21
S7_hoja_control,10,12,14,1,1.20


### Observaciones de la Parte A

*(A completar tras ejecutar: comentar el ratio tokens/palabra, qué palabras se fragmentan y por qué —tildes, morfología rica, baja frecuencia en el vocabulario multilingüe— y qué implica para leer la matriz de atención en la Parte C.)*

---
# Parte B — Ejecución del modelo con `output_attentions=True`

El modelo se carga en modo evaluación y se le pide que devuelva las atenciones. `outputs.attentions` es una **tupla de 12 tensores**, uno por capa; cada tensor tiene forma

```
(batch_size, num_heads, num_tokens, num_tokens)
```

La entrada `[capa][lote, cabeza, i, j]` es el peso con que el token de la posición `i` (la *consulta*) atiende al token de la posición `j` (la *clave*). Cada fila `i` es una distribución de probabilidad: **suma 1** sobre `j`.

In [10]:
modelo = AutoModel.from_pretrained(NOMBRE_MODELO, output_attentions=True)
modelo.eval()

cfg = modelo.config
print("Modelo            :", NOMBRE_MODELO)
print("Capas ocultas     :", cfg.num_hidden_layers)
print("Cabezas por capa  :", cfg.num_attention_heads)
print("Dim. oculta       :", cfg.hidden_size)
print("Dim. por cabeza   :", cfg.hidden_size // cfg.num_attention_heads)
print("Parámetros        :", f"{sum(p.numel() for p in modelo.parameters()):,}")


Loading weights:   0%|                                 | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 21226.45it/s]


[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modelo            : bert-base-multilingual-cased
Capas ocultas     : 12
Cabezas por capa  : 12
Dim. oculta       : 768
Dim. por cabeza   : 64
Parámetros        : 177,853,440


In [11]:
def ejecutar(texto):
    """Tokeniza, ejecuta el modelo y devuelve (tokens, attentions).

    attentions: tupla de longitud n_capas; cada elemento tiene forma
                (batch, n_cabezas, n_tokens, n_tokens).
    """
    entradas = tokenizador(texto, return_tensors="pt")
    salidas = modelo(**entradas)
    tokens = tokenizador.convert_ids_to_tokens(entradas["input_ids"][0])
    return tokens, salidas.attentions


# Ejecutamos todas las oraciones y guardamos el resultado para las Partes C y D
RESULTADOS = {}
for oid, info in ORACIONES.items():
    tokens, attn = ejecutar(info["texto"])
    RESULTADOS[oid] = {"tokens": tokens, "attentions": attn}

print(f"Oraciones procesadas: {len(RESULTADOS)}")

Oraciones procesadas: 7


In [12]:
# B.1 — Forma de los tensores de atención (ejemplo detallado sobre una oración)
oid = "S3_sub"
tokens = RESULTADOS[oid]["tokens"]
attn = RESULTADOS[oid]["attentions"]

print(f"Oración: «{ORACIONES[oid]['texto']}»")
print(f"Tokens ({len(tokens)}): {tokens}")
print()
print(f"type(outputs.attentions) = {type(attn).__name__}")
print(f"len(outputs.attentions)  = {len(attn)}   <- una entrada por capa")
print(f"attentions[0].shape      = {tuple(attn[0].shape)}")
print("                            (batch_size, num_heads, num_tokens, num_tokens)")
print()
print("Todas las capas tienen la misma forma:",
      all(a.shape == attn[0].shape for a in attn))

Oración: «Gregorio oía tras de sí una voz que parecía imposible que fuese la de un padre.»
Tokens (22): ['[CLS]', 'Gregorio', 'o', '##ía', 'tras', 'de', 'sí', 'una', 'voz', 'que', 'pare', '##cía', 'imposible', 'que', 'fue', '##se', 'la', 'de', 'un', 'padre', '.', '[SEP]']

type(outputs.attentions) = tuple
len(outputs.attentions)  = 12   <- una entrada por capa
attentions[0].shape      = (1, 12, 22, 22)
                            (batch_size, num_heads, num_tokens, num_tokens)

Todas las capas tienen la misma forma: True


In [13]:
# B.2 — Tabla de dimensiones para todas las oraciones
filas = []
for oid, r in RESULTADOS.items():
    a0 = r["attentions"][0]
    b, h, n, m = a0.shape
    filas.append({
        "oración": oid,
        "n_capas": len(r["attentions"]),
        "batch": b,
        "cabezas": h,
        "tokens (filas)": n,
        "tokens (columnas)": m,
        "shape por capa": str(tuple(a0.shape)),
        "matrices de atención": len(r["attentions"]) * h,
    })

pd.DataFrame(filas).set_index("oración")

,n_capas,batch,cabezas,tokens (filas),tokens (columnas),shape por capa,matrices de atención
oración,,,,,,,
S1_simple,12,1,12,14,14,"(1, 12, 14, 14)",144
S2_coord,12,1,12,25,25,"(1, 12, 25, 25)",144
S3_sub,12,1,12,22,22,"(1, 12, 22, 22)",144
S4_larga,12,1,12,51,51,"(1, 12, 51, 51)",144
S5_hoja_puerta,12,1,12,16,16,"(1, 12, 16, 16)",144
S6_hojas_puerta,12,1,12,19,19,"(1, 12, 19, 19)",144
S7_hoja_control,12,1,12,14,14,"(1, 12, 14, 14)",144


In [14]:
# B.3 — Comprobación: cada fila es una distribución de probabilidad (suma 1)
a = RESULTADOS["S3_sub"]["attentions"][5][0, 3]     # capa 6, cabeza 4
sumas = a.sum(dim=-1)
print("Forma de la matriz (capa 6, cabeza 4):", tuple(a.shape))
print("Suma de cada fila:", [round(float(s), 4) for s in sumas])
print("¿Todas ≈ 1.0?", bool(torch.allclose(sumas, torch.ones_like(sumas), atol=1e-4)))

Forma de la matriz (capa 6, cabeza 4): (22, 22)
Suma de cada fila: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
¿Todas ≈ 1.0? True


---
# Parte C — Análisis de atención

Se comparan las **capas 3 y 12** y las **cabezas 2 y 8** (en los índices de Python: capas `2` y `11`, cabezas `1` y `7`). Para cada oración se toman las dos primeras palabras de `objetivos`; así se cumplen dos capas, dos cabezas y dos tokens lingüísticamente relevantes por oración.

Cuando una palabra objetivo ocupa varias subpalabras WordPiece, la fila de consulta se obtiene promediando las filas de todas sus piezas. Los destinos siguen siendo tokens del modelo: por eso cada resultado informa tanto el token como su posición y conserva por separado piezas repetidas. Los tokens especiales no se eliminan, pues también forman parte real de la matriz de atención.

In [15]:
# C.1 — Selecciones y funciones auxiliares
CAPAS = (2, 11)      # capas 3 y 12 en numeración humana
CABEZAS = (1, 7)    # cabezas 2 y 8 en numeración humana
K = 5

# Se usan exactamente dos objetivos por oración para mantener legible la salida.
OBJETIVOS_C = {oid: info["objetivos"][:2] for oid, info in ORACIONES.items()}

def buscar_subsecuencia(secuencia, patron):
    """Índices de la primera aparición contigua de patron en secuencia."""
    for inicio in range(len(secuencia) - len(patron) + 1):
        if secuencia[inicio:inicio + len(patron)] == patron:
            return list(range(inicio, inicio + len(patron)))
    raise ValueError(f"No se encontró {patron} en {secuencia}")

def posiciones_objetivo(tokens, palabra):
    """Localiza todas las piezas WordPiece de una palabra objetivo."""
    piezas = tokenizador.tokenize(palabra)
    return buscar_subsecuencia(tokens, piezas), piezas

def top_atencion(oid, palabra, capa, cabeza, k=5):
    """Cinco claves con mayor atención desde una palabra objetivo.

    Si la consulta se fragmenta, promedia sus filas de atención.
    """
    r = RESULTADOS[oid]
    posiciones, piezas = posiciones_objetivo(r["tokens"], palabra)
    matriz = r["attentions"][capa][0, cabeza]
    fila = matriz[posiciones].mean(dim=0)
    valores, destinos = torch.topk(fila, k=min(k, fila.numel()))

    filas = []
    for rango, (peso, destino) in enumerate(zip(valores, destinos), start=1):
        j = int(destino)
        filas.append({
            "oración": oid,
            "objetivo": palabra,
            "piezas_objetivo": " + ".join(piezas),
            "pos_objetivo": " + ".join(map(str, posiciones)),
            "capa": capa + 1,
            "cabeza": cabeza + 1,
            "rango": rango,
            "token_atendido": r["tokens"][j],
            "pos_atendida": j,
            "peso": float(peso),
        })
    return filas

print("Capas (numeración humana):", [c + 1 for c in CAPAS])
print("Cabezas (numeración humana):", [h + 1 for h in CABEZAS])
pd.Series({oid: ", ".join(objs) for oid, objs in OBJETIVOS_C.items()}, name="objetivos")

Capas (numeración humana): [3, 12]
Cabezas (numeración humana): [2, 8]


S1_simple          incrustó, puerta
S2_coord              quedó, umbral
S3_sub                   voz, padre
S4_larga              patas, herida
S5_hoja_puerta          hoja, abrir
S6_hojas_puerta       hojas, puerta
S7_hoja_control         hoja, árbol
Name: objetivos, dtype: str

In [16]:
# C.2 — Cálculo de los cinco tokens con mayor peso para cada selección
filas_top5 = []
for oid, objetivos in OBJETIVOS_C.items():
    for objetivo in objetivos:
        for capa in CAPAS:
            for cabeza in CABEZAS:
                filas_top5.extend(top_atencion(oid, objetivo, capa, cabeza, K))

df_top5 = pd.DataFrame(filas_top5)
df_top5["peso"] = df_top5["peso"].round(6)

# Comprobaciones explícitas de los requisitos de la Parte C.
assert len(CAPAS) >= 2 and len(set(CAPAS)) == len(CAPAS)
assert len(CABEZAS) >= 2 and len(set(CABEZAS)) == len(CABEZAS)
assert all(len(objetivos) >= 2 for objetivos in OBJETIVOS_C.values())
conteos = df_top5.groupby(["oración", "objetivo", "capa", "cabeza"]).size()
assert (conteos == K).all()

print(f"Selecciones analizadas: {len(conteos)}")
print(f"Filas producidas: {len(df_top5)} ({K} tokens por selección)")
print("Todos los grupos tienen exactamente cinco tokens:", bool((conteos == K).all()))

Selecciones analizadas: 56
Filas producidas: 280 (5 tokens por selección)
Todos los grupos tienen exactamente cinco tokens: True


### C.3 Resultados

Cada bloque siguiente muestra los cinco destinos de mayor peso para cada combinación de palabra objetivo, capa y cabeza. `rango = 1` corresponde al peso máximo. Si aparece `[CLS]`, `[SEP]` o puntuación, se conserva como evidencia y no se interpreta automáticamente como una relación lingüística.

In [17]:
# Una tabla por oración evita una única salida difícil de leer.
columnas_c = [
    "objetivo", "piezas_objetivo", "pos_objetivo",
    "capa", "cabeza", "rango", "token_atendido",
    "pos_atendida", "peso",
]
for oid in ORACIONES:
    print("=" * 100)
    print(f"{oid}: «{ORACIONES[oid]['texto']}»")
    display(df_top5.loc[df_top5["oración"] == oid, columnas_c].reset_index(drop=True))

S1_simple: «Se incrustó en el marco de la puerta.»


,objetivo,piezas_objetivo,pos_objetivo,capa,cabeza,rango,token_atendido,pos_atendida,peso
0,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,2,1,[CLS],0,0.355321
1,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,2,2,.,12,0.148458
2,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,2,3,[SEP],13,0.098415
3,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,2,4,marco,8,0.074513
4,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,2,5,en,6,0.066231
5,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,8,1,in,2,0.240893
6,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,8,2,[CLS],0,0.203884
7,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,8,3,##rust,4,0.188894
8,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,8,4,Se,1,0.188308
9,incrustó,in + ##c + ##rust + ##ó,2 + 3 + 4 + 5,3,8,5,##c,3,0.171514


S2_coord: «Se irguió de medio lado y quedó atravesado en el umbral, lacerándose el costado.»


,objetivo,piezas_objetivo,pos_objetivo,capa,cabeza,rango,token_atendido,pos_atendida,peso
0,quedó,quedó,9,3,2,1,[CLS],0,0.170503
1,quedó,quedó,9,3,2,2,",",16,0.142266
2,quedó,quedó,9,3,2,3,umbral,15,0.110861
3,quedó,quedó,9,3,2,4,en,13,0.093480
4,quedó,quedó,9,3,2,5,##sado,12,0.063550
5,quedó,quedó,9,3,8,1,y,8,0.564456
6,quedó,quedó,9,3,8,2,[CLS],0,0.379185
7,quedó,quedó,9,3,8,3,quedó,9,0.029293
8,quedó,quedó,9,3,8,4,lado,7,0.007743
9,quedó,quedó,9,3,8,5,##rave,11,0.007592


S3_sub: «Gregorio oía tras de sí una voz que parecía imposible que fuese la de un padre.»


...

,objetivo,piezas_objetivo,pos_objetivo,capa,cabeza,rango,token_atendido,pos_atendida,peso
0,hoja,ho + ##ja,8 + 9,3,2,1,[CLS],0,0.439675
1,hoja,ho + ##ja,8 + 9,3,2,2,.,14,0.132822
2,hoja,ho + ##ja,8 + 9,3,2,3,[SEP],15,0.118006
3,hoja,ho + ##ja,8 + 9,3,2,4,espacio,12,0.077782
4,hoja,ho + ##ja,8 + 9,3,2,5,suficiente,13,0.069785
5,hoja,ho + ##ja,8 + 9,3,8,1,ho,8,0.491197
6,hoja,ho + ##ja,8 + 9,3,8,2,[CLS],0,0.282267
7,hoja,ho + ##ja,8 + 9,3,8,3,otra,7,0.204019
8,hoja,ho + ##ja,8 + 9,3,8,4,##ja,9,0.009671
9,hoja,ho + ##ja,8 + 9,3,8,5,la,6,0.005422


S6_hojas_puerta: «Durante la noche se entreabrió una vez una de las hojas de la puerta.»


,objetivo,piezas_objetivo,pos_objetivo,capa,cabeza,rango,token_atendido,pos_atendida,peso
0,hojas,hojas,13,3,2,1,[CLS],0,0.421082
1,hojas,hojas,13,3,2,2,.,17,0.186717
2,hojas,hojas,13,3,2,3,puerta,16,0.134578
3,hojas,hojas,13,3,2,4,[SEP],18,0.070826
4,hojas,hojas,13,3,2,5,la,15,0.041868
5,hojas,hojas,13,3,8,1,[CLS],0,0.483906
6,hojas,hojas,13,3,8,2,las,12,0.472243
7,hojas,hojas,13,3,8,3,hojas,13,0.025170
8,hojas,hojas,13,3,8,4,de,11,0.007403
9,hojas,hojas,13,3,8,5,[SEP],18,0.003432


S7_hoja_control: «La hoja del árbol cayó sobre el suelo del jardín.»


,objetivo,piezas_objetivo,pos_objetivo,capa,cabeza,rango,token_atendido,pos_atendida,peso
0,hoja,ho + ##ja,2 + 3,3,2,1,[CLS],0,0.354875
1,hoja,ho + ##ja,2 + 3,3,2,2,.,12,0.113960
2,hoja,ho + ##ja,2 + 3,3,2,3,[SEP],13,0.096351
3,hoja,ho + ##ja,2 + 3,3,2,4,cayó,6,0.094291
4,hoja,ho + ##ja,2 + 3,3,2,5,sobre,7,0.090228
5,hoja,ho + ##ja,2 + 3,3,8,1,ho,2,0.459710
6,hoja,ho + ##ja,2 + 3,3,8,2,[CLS],0,0.334647
7,hoja,ho + ##ja,2 + 3,3,8,3,La,1,0.186105
8,hoja,ho + ##ja,2 + 3,3,8,4,##ja,3,0.012117
9,hoja,ho + ##ja,2 + 3,3,8,5,árbol,5,0.001958


---
# Parte D — Comparación entre contextos

Se contrasta la palabra polisémica **hoja** en dos oraciones: `S5_hoja_puerta`, donde significa *batiente de una puerta*, y `S7_hoja_control`, donde significa *hoja vegetal*. Se mantienen constantes el modelo, las capas y las cabezas; cambia el contexto de la palabra.

La primera tabla coloca lado a lado los cinco tokens más atendidos. La segunda cuantifica la coincidencia entre ambos conjuntos mediante Jaccard (`0` = ningún token compartido; `1` = los mismos tokens). Como las oraciones tienen vocabularios diferentes, esta medida es descriptiva y no constituye por sí sola una prueba de desambiguación semántica.

In [18]:
# D.1 — Comparación directa de los top 5 desde 'hoja'
PAR_HOJA = ("S5_hoja_puerta", "S7_hoja_control")
df_hoja = df_top5[
    df_top5["oración"].isin(PAR_HOJA) & (df_top5["objetivo"] == "hoja")
].copy()

comparacion_top = (
    df_hoja.assign(destino=lambda x: x["token_atendido"] + " (" + x["peso"].map(lambda v: f"{v:.4f}") + ")")
    .pivot(index=["capa", "cabeza", "rango"], columns="oración", values="destino")
    .reset_index()
)
display(comparacion_top)

filas_solapamiento = []
for capa in (c + 1 for c in CAPAS):
    for cabeza in (h + 1 for h in CABEZAS):
        conjuntos = {}
        for oid in PAR_HOJA:
            sel = df_hoja[(df_hoja["oración"] == oid) &
                           (df_hoja["capa"] == capa) &
                           (df_hoja["cabeza"] == cabeza)]
            conjuntos[oid] = set(sel["token_atendido"])
        a, b = (conjuntos[oid] for oid in PAR_HOJA)
        union = a | b
        filas_solapamiento.append({
            "capa": capa,
            "cabeza": cabeza,
            "tokens_compartidos": ", ".join(sorted(a & b)) or "—",
            "n_compartidos": len(a & b),
            "Jaccard_top5": round(len(a & b) / len(union), 3),
            "¿cambia_el_top5?": a != b,
        })

df_solapamiento = pd.DataFrame(filas_solapamiento)
df_solapamiento

oración,capa,cabeza,rango,S5_hoja_puerta,S7_hoja_control
0,3,2,1,[CLS] (0.4397),[CLS] (0.3549)
1,3,2,2,. (0.1328),. (0.1140)
2,3,2,3,[SEP] (0.1180),[SEP] (0.0964)
3,3,2,4,espacio (0.0778),cayó (0.0943)
4,3,2,5,suficiente (0.0698),sobre (0.0902)
5,3,8,1,ho (0.4912),ho (0.4597)
6,3,8,2,[CLS] (0.2823),[CLS] (0.3346)
7,3,8,3,otra (0.2040),La (0.1861)
8,3,8,4,##ja (0.0097),##ja (0.0121)
9,3,8,5,la (0.0054),árbol (0.0020)


,capa,cabeza,tokens_compartidos,n_compartidos,Jaccard_top5,¿cambia_el_top5?
0,3,2,"., [CLS], [SEP]",3,0.429,True
1,3,8,"##ja, [CLS], ho",3,0.429,True
2,12,2,"##ja, ., [CLS], ho",4,0.667,True
3,12,8,ho,1,0.111,True


In [19]:
# D.2 — Peso dirigido desde 'hoja' hacia palabras indicativas de cada sentido
PISTAS_CONTEXTO = {
    "S5_hoja_puerta": ["abrir", "espacio", "suficiente"],
    "S7_hoja_control": ["árbol", "cayó", "suelo", "jardín"],
}

def peso_entre_palabras(oid, consulta, clave, capa, cabeza):
    """Suma el peso dirigido a todas las piezas de clave."""
    r = RESULTADOS[oid]
    pos_q, _ = posiciones_objetivo(r["tokens"], consulta)
    pos_k, piezas_k = posiciones_objetivo(r["tokens"], clave)
    fila = r["attentions"][capa][0, cabeza, pos_q].mean(dim=0)
    return float(fila[pos_k].sum()), " + ".join(piezas_k)

filas_pistas = []
for oid, pistas in PISTAS_CONTEXTO.items():
    for capa in CAPAS:
        for cabeza in CABEZAS:
            for pista in pistas:
                peso, piezas = peso_entre_palabras(oid, "hoja", pista, capa, cabeza)
                filas_pistas.append({
                    "oración": oid, "capa": capa + 1, "cabeza": cabeza + 1,
                    "pista_contextual": pista, "piezas_pista": piezas, "peso": peso,
                })

df_pistas = pd.DataFrame(filas_pistas)
df_pistas["peso"] = df_pistas["peso"].round(6)
display(df_pistas.sort_values(["capa", "cabeza", "oración", "peso"], ascending=[True, True, True, False]))

print("Conclusión automática del contraste:")
if df_solapamiento["¿cambia_el_top5?"].all():
    print("Sí: el conjunto de cinco destinos cambia en todas las capas y cabezas seleccionadas.")
elif df_solapamiento["¿cambia_el_top5?"].any():
    print("Sí, parcialmente: cambia en algunas combinaciones de capa y cabeza.")
else:
    print("No se observa cambio en los conjuntos top 5 seleccionados.")
print("Las pistas con mayor peso en cada contexto son:")
idx = df_pistas.groupby(["oración", "capa", "cabeza"])["peso"].idxmax()
display(df_pistas.loc[idx, ["oración", "capa", "cabeza", "pista_contextual", "peso"]]
        .sort_values(["capa", "cabeza", "oración"]).reset_index(drop=True))

,oración,capa,cabeza,pista_contextual,piezas_pista,peso
1,S5_hoja_puerta,3,2,espacio,espacio,0.077782
2,S5_hoja_puerta,3,2,suficiente,suficiente,0.069785
0,S5_hoja_puerta,3,2,abrir,abrir,0.009244
13,S7_hoja_control,3,2,cayó,cayó,0.094291
12,S7_hoja_control,3,2,árbol,árbol,0.067126
14,S7_hoja_control,3,2,suelo,suelo,0.040197
15,S7_hoja_control,3,2,jardín,jardín,0.038438
3,S5_hoja_puerta,3,8,abrir,abrir,0.001297
5,S5_hoja_puerta,3,8,suficiente,suficiente,0.000078
4,S5_hoja_puerta,3,8,espacio,espacio,0.000010


Conclusión automática del contraste:
Sí: el conjunto de cinco destinos cambia en todas las capas y cabezas seleccionadas.
Las pistas con mayor peso en cada contexto son:


,oración,capa,cabeza,pista_contextual,peso
0,S5_hoja_puerta,3,2,espacio,0.077782
1,S7_hoja_control,3,2,cayó,0.094291
2,S5_hoja_puerta,3,8,abrir,0.001297
3,S7_hoja_control,3,8,árbol,0.001958
4,S5_hoja_puerta,12,2,abrir,0.009848
5,S7_hoja_control,12,2,cayó,0.021822
6,S5_hoja_puerta,12,8,abrir,0.075998
7,S7_hoja_control,12,8,árbol,0.018548


### Interpretación de la comparación

Sí cambia la atención cuando cambia el contexto: `¿cambia_el_top5?` es verdadero en las cuatro combinaciones y los índices Jaccard son `0.429`, `0.429`, `0.667` y `0.111`, nunca `1`. En la capa 3, cabeza 2, ambos contextos comparten `[CLS]`, `[SEP]` y el punto, pero los tokens de contenido cambian de **espacio/suficiente** a **cayó/sobre**. En la capa 12, cabeza 8, el solapamiento es mínimo: solamente se comparte la pieza `ho`.

La tabla de pistas refuerza el contraste: según la combinación, la pista principal del sentido arquitectónico es **espacio** o **abrir**, mientras que en el sentido vegetal es **cayó** o **árbol**. Por ejemplo, en capa 12/cabeza 8, *hoja* dirige `0.075998` a *abrir* en la primera oración y `0.018548` a *árbol* en la segunda. Las variaciones entre las cuatro combinaciones muestran que no existe un único patrón de atención. Esta evidencia describe un cambio contextual en los pesos, pero no basta para afirmar que una cabeza aislada representa el significado completo de la palabra ni que la atención sea una explicación causal de la predicción del modelo.

---
# 6. Preguntas de análisis — para completar


1. **¿Qué tokens reciben mayor atención desde cada token seleccionado?**

Los tokens con mayor atencion dependen de la capa y la cabez, en la oracion simple atienden principalmente a los CLS como lo son a sus propias subpalabras. (in, ##c, ##rust, ##ó) y a marco; puerta atiende a [CLS], la, puerta y marco

2. **¿Cambian los patrones entre capas?**

Lo que se vio que en la capa 3 si aparece mucha atencion hacia CLS, SEP la puntuacion , los determinantes y algunas palabras si cambiaron. En cambio en la capa 12 es mas visto que los tokens se atiendan a ellos o a otras palabras que los forman. Esto indica que entre capas no conservan relacion

3. **¿Cambian los patrones entre cabezas?**

Si es asi porque las cabezas muesttran comportamientos diferentes incluso dentro de una misma capa. La cabeza 2 atiende con frecuencia a CLS, la puntuacion y el propio token al igual que palabras de contenido. La 8 solo se concentra en determinantes o palabras casi iguales, o subpalabras. 

4. **¿Las palabras con mayor atención son lingüísticamente relevantes?**

Esto no mucho pero si en parte, por ejemplo : Herida , accidente . O otro ejemplo hojas , puerta, tambien otro que es arbol , jardin. Sin embargo entre los valores mas altos tambien aparecen CLS, SEP, la puntuacion del token. Por ello si tienen mucho peso no quiere decir que tengan sentido en su relacion en el lenguaje. 

5. **¿Qué diferencias aparecen entre oraciones simples y complejas?**

Las simples tienen relaciones locales , como puerta y marco, y atencion hacia el propio token, en las complejas hay conexiones mas alejadas, por ejemplo voz atiende a imposible y padre atiende a voz. Pero entre mas compleja no es lo unico que influye sino tambien la capa que estemos .

6. **¿Qué ocurre cuando una palabra se divide en subpalabras?**

Pasa a tener varias lugares en la matriz de atencion. Por ejemplo hoja se presenta como ho + ##ja, por ello las piezas pueden prestarse atención entre sí y aparecer por separado entre los cinco valores máximos. En este análisis se promediaron las filas de todas las subpalabras para obtener una sola distribución desde la palabra completa. Esto evita elegir arbitrariamente una única pieza

7. **¿Qué no puedes concluir observando únicamente los pesos de atención?**

No se puede concluir que :
- EL modelo comprenda que dijimos en una oracion.
- Que una cabeza representa una regla gramatical o significado
